In [0]:
%sql
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file

In [0]:
%sql
select * from com_edp_prd.com_intgr.distribution_sd_shipments
where is_current = true

In [0]:
%sql
select * from com_edp_prd.com_intgr.distribution_accounts 

In [0]:
%sql
select distinct crx_account_id 
-- quantity_shipped,order_date 
from com_edp_prd.com_intgr.distribution_sd_shipments
where quantity_shipped > 0
-- order by order_date desc

In [0]:
%sql
select count(patient_id) from com_edp_prd.cmpa_insights_internal_schema.patient360_master
where latest_mpsii_tx_type is null

In [0]:
%sql
select crx_account_id,crx_account_name,invoice_number,order_date,invoicedate,ndc,quantity_shipped,return_quantity,sd_name,ingestion_date from com_edp_prd.com_intgr.distribution_sd_shipments

In [0]:
%sql
select * from com_edp_prd.com_intgr.distribution_sd_shipments

In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level AS

WITH shipments_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY invoice_number
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE is_current = true
          AND ndc = '84976-0001-01'
    )
    WHERE rn = 1
),

accounts_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_accounts
        WHERE is_current = true
    )
    WHERE rn = 1
),

zip_territory AS (
    SELECT DISTINCT
        zipcode,
        territory_id,
        territory_name,
        region_id,
        region_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
),

base AS (
    SELECT
        s.crx_account_id,
        a.account_facility_name,
        a.account_type,
        s.order_date,
        s.quantity_shipped,
        s.ship_to_address_postal_code,
        z.territory_id,
        z.territory_name,
        z.region_id,
        z.region_name
    FROM shipments_dedup s
    LEFT JOIN accounts_dedup a
        ON s.crx_account_id = a.crx_account_id
    LEFT JOIN zip_territory z
        ON SUBSTRING(s.ship_to_address_postal_code, 1, 5) = z.zipcode
)

SELECT
    crx_account_id,
    account_facility_name,
    CASE 
    WHEN UPPER(account_facility_name) LIKE '%ORSINI%' THEN 'SP'
    ELSE 'HCO'
    END AS account_type,
    substring(ship_to_address_postal_code, 1, 5) AS postal_code,
    territory_id,
    territory_name,
    region_id,
    region_name,

    /* LTD from March 1, 2026 */
    SUM(CASE 
        WHEN order_date >= DATE('2026-03-01') 
        THEN quantity_shipped 
    END) AS LTD,

    /* MTD */
    SUM(CASE 
        WHEN order_date >= DATE_TRUNC('month', CURRENT_DATE) 
        THEN quantity_shipped 
    END) AS MTD,

    /* QTD */
    SUM(CASE 
        WHEN order_date >= DATE_TRUNC('quarter', CURRENT_DATE) 
        THEN quantity_shipped 
    END) AS QTD,

    /* YTD */
    SUM(CASE 
        WHEN order_date >= DATE_TRUNC('year', CURRENT_DATE) 
        THEN quantity_shipped 
    END) AS YTD

FROM base
GROUP BY 1,2,3,4,5,6,7,8;

In [0]:
%sql
Select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
where zipcode in 
-- ('53226','28203','52242','60007','15224')
('64874','35812','21009','72516','41334')

In [0]:
%sql
select * from com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level

In [0]:
%sql
SELECT
    SUM(quantity_shipped) AS raw_qty
FROM com_edp_prd.com_intgr.distribution_sd_shipments
WHERE is_current = true;

SELECT
    SUM(quantity_shipped) AS dedup_qty
FROM (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY invoice_number
                   ORDER BY ingestion_date DESC
               ) rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE is_current = true
    ) WHERE rn = 1
);

In [0]:
%sql
select * from com_edp_prd.com_intgr.distribution_accounts

In [0]:
%sql
Select * from com_edp_prd.com_intgr.sp_dispense

In [0]:
%sql
select * from com_edp_prd.com_intgr.payer_policy_details